In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

In [ ]:
data = load_wine()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
df['class'] = df['target'].map({0: 'Class_0', 1: 'Class_1', 2: 'Class_2'})

In [ ]:
print('Shape:', df.shape)
print('\nData Types:')
print(df.dtypes)
print('\nMissing Values:', df.isnull().sum().sum())
print('\nClass Distribution:')
print(df['class'].value_counts())
print('\nDescriptive Statistics:')
df.describe().round(4)

In [ ]:
COLORS = ['#4A90D9', '#E05C5C', '#27AE60']
TEXT = '#2C3E50'
BG = '#F4F6F9'
GRID = '#D5D8DC'

plt.rcParams.update({
    'axes.facecolor': BG,
    'figure.facecolor': 'white',
    'axes.edgecolor': GRID,
    'axes.grid': True,
    'grid.color': GRID,
    'grid.linewidth': 0.6,
    'text.color': TEXT,
    'axes.labelcolor': TEXT,
    'xtick.color': TEXT,
    'ytick.color': TEXT
})

In [ ]:
counts = df['class'].value_counts()

fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autos = ax.pie(
    counts.values, labels=counts.index, colors=COLORS,
    autopct='%1.1f%%', startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=2),
    textprops={'fontsize': 11}
)
for at in autos:
    at.set_fontsize(12)
    at.set_fontweight('bold')
    at.set_color('white')
ax.set_title('Wine Dataset - Class Distribution', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(counts.index, counts.values, color=COLORS, edgecolor='white', linewidth=1.5, width=0.5)
for b in bars:
    ax.text(
        b.get_x() + b.get_width() / 2,
        b.get_height() + 0.8,
        str(int(b.get_height())),
        ha='center', va='bottom', fontweight='bold', fontsize=13
    )
ax.set_ylim(0, max(counts.values) * 1.2)
ax.set_title('Wine Dataset - Sample Count per Class', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Samples')
ax.grid(axis='x', visible=False)
plt.tight_layout()
plt.show()

In [ ]:
X_all = df.drop(columns=['target', 'class']).values
y_all = df['target'].values

fig, ax = plt.subplots(figsize=(10, 5))
class_labels = ['Class_0', 'Class_1', 'Class_2']
for i, (cls, col) in enumerate(zip([0, 1, 2], COLORS)):
    mask = y_all == cls
    ax.plot(
        np.where(mask)[0], X_all[mask, 0], 'o',
        color=col, markersize=4, alpha=0.75, label=class_labels[i]
    )
ax.set_title('Alcohol Feature Value by Class (Line Chart)', fontsize=14, fontweight='bold')
ax.set_xlabel('Sample Index')
ax.set_ylabel('Alcohol')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
X = df.drop(columns=['target', 'class']).values.astype(float)
y = df['target'].values.astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print('Training samples:', X_train_sc.shape[0])
print('Testing  samples:', X_test_sc.shape[0])

In [ ]:
model = LogisticRegression(max_iter=10000, solver='lbfgs', random_state=42)
model.fit(X_train_sc, y_train)

y_train_pred = model.predict(X_train_sc)
y_test_pred = model.predict(X_test_sc)

In [ ]:
def get_metrics(y_true, y_pred, label):
    return {
        'Set': label,
        'Accuracy': round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred, average='weighted', zero_division=0), 4),
        'Recall': round(recall_score(y_true, y_pred, average='weighted'), 4),
        'F1-Score': round(f1_score(y_true, y_pred, average='weighted'), 4),
    }

train_m = get_metrics(y_train, y_train_pred, 'Training')
test_m = get_metrics(y_test, y_test_pred, 'Testing')

print('Training Results:')
print(classification_report(y_train, y_train_pred, target_names=['Class_0', 'Class_1', 'Class_2']))
print('Testing Results:')
print(classification_report(y_test, y_test_pred, target_names=['Class_0', 'Class_1', 'Class_2']))

In [ ]:
cm_tr = confusion_matrix(y_train, y_train_pred)

def draw_cm(cm, title, cmap):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap=cmap)
    ax.set_xticks([0, 1, 2])
    ax.set_yticks([0, 1, 2])
    ax.set_xticklabels(['Pred C0', 'Pred C1', 'Pred C2'])
    ax.set_yticklabels(['Act C0', 'Act C1', 'Act C2'])
    thresh = cm.max() / 2.0
    for i in range(3):
        for j in range(3):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    fontsize=18, fontweight='bold',
                    color='white' if cm[i, j] > thresh else TEXT)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
    plt.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.show()

draw_cm(cm_tr, 'Confusion Matrix - Training Set', 'Blues')

In [ ]:
cm_te = confusion_matrix(y_test, y_test_pred)
draw_cm(cm_te, 'Confusion Matrix - Testing Set', 'Reds')

In [ ]:
metric_keys = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
tr_vals = [train_m[k] for k in metric_keys]
te_vals = [test_m[k] for k in metric_keys]
x_pos = np.arange(len(metric_keys))
w = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x_pos - w/2, tr_vals, w, label='Training', color='#4A90D9', edgecolor='white')
ax.bar(x_pos + w/2, te_vals, w, label='Testing', color='#E05C5C', edgecolor='white')
ax.set_xticks(x_pos)
ax.set_xticklabels(metric_keys, fontsize=11)
ax.set_ylim(0.85, 1.05)
ax.set_title('Metrics Comparison - Training vs Testing', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
for xi, (tv, ev) in enumerate(zip(tr_vals, te_vals)):
    ax.text(xi - w/2, tv + 0.003, f'{tv:.3f}', ha='center', fontsize=9, fontweight='bold')
    ax.text(xi + w/2, ev + 0.003, f'{ev:.3f}', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
feat_names = list(data.feature_names)
coef_abs = np.mean(np.abs(model.coef_), axis=0)
top_idx = np.argsort(coef_abs)[::-1]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(
    [feat_names[i] for i in top_idx][::-1],
    coef_abs[top_idx][::-1],
    color='#4A90D9', edgecolor='white'
)
ax.set_title('Feature Importances - Mean Absolute Coefficient', fontsize=13, fontweight='bold')
ax.set_xlabel('Mean Absolute Coefficient Value')
plt.tight_layout()
plt.show()

In [ ]:
def cm_parts(cm):
    tp = int(np.diag(cm).sum())
    fp = int((cm.sum(axis=0) - np.diag(cm)).sum())
    fn = int((cm.sum(axis=1) - np.diag(cm)).sum())
    tn = int(cm.sum() - tp - fp - fn)
    return tp, tn, fp, fn

tp_tr, tn_tr, fp_tr, fn_tr = cm_parts(cm_tr)
tp_te, tn_te, fp_te, fn_te = cm_parts(cm_te)

results_df = pd.DataFrame([
    {
        'Dataset': 'Training',
        'Accuracy': f"{train_m['Accuracy']:.4f}",
        'Precision': f"{train_m['Precision']:.4f}",
        'Recall': f"{train_m['Recall']:.4f}",
        'F1-Score': f"{train_m['F1-Score']:.4f}",
        'TP': tp_tr, 'TN': tn_tr, 'FP': fp_tr, 'FN': fn_tr
    },
    {
        'Dataset': 'Testing',
        'Accuracy': f"{test_m['Accuracy']:.4f}",
        'Precision': f"{test_m['Precision']:.4f}",
        'Recall': f"{test_m['Recall']:.4f}",
        'F1-Score': f"{test_m['F1-Score']:.4f}",
        'TP': tp_te, 'TN': tn_te, 'FP': fp_te, 'FN': fn_te
    }
])

results_df.set_index('Dataset', inplace=True)
print('Final Results Table')
print('='*65)
results_df

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))
ax.axis('off')

col_labels = ['Dataset', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'TP', 'TN', 'FP', 'FN']
cell_text = [
    ['Training', f"{train_m['Accuracy']:.4f}", f"{train_m['Precision']:.4f}",
     f"{train_m['Recall']:.4f}", f"{train_m['F1-Score']:.4f}",
     tp_tr, tn_tr, fp_tr, fn_tr],
    ['Testing', f"{test_m['Accuracy']:.4f}", f"{test_m['Precision']:.4f}",
     f"{test_m['Recall']:.4f}", f"{test_m['F1-Score']:.4f}",
     tp_te, tn_te, fp_te, fn_te]
]

tbl = ax.table(
    cellText=cell_text,
    colLabels=col_labels,
    cellLoc='center',
    loc='center',
    bbox=[0, 0, 1, 1]
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(12)

for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor(GRID)
    cell.set_height(0.35)
    if r == 0:
        cell.set_facecolor('#2C3E50')
        cell.set_text_props(color='white', fontweight='bold')
    elif r == 1:
        cell.set_facecolor('#D6EAF8')
    else:
        cell.set_facecolor('#D5F5E3')

ax.set_title('Results Table - Training vs Testing', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()